# SO(n) Lie Algebra Watermarking: z 기반 회전 + Full SO(n) 표현력

기존 SO(n) 접근의 2가지 근본 한계를 동시에 해결합니다:

| | 기존 SO(n) | 본 노트북 |
|---|---|---|
| 출발점 | μ (고정, z 정보 손실) | **z (원본 정보 보존)** |
| 회전 자유도 | 100개 Givens angles | **Lie algebra (rank-2k 자유도)** |
| 비트 상호작용 | 없음 or MLP→angles | **반대칭 행렬로 전체 상호작용** |
| 구면 보장 | Givens (자동) | **Cayley transform (자동)** |

**핵심 아이디어:**
1. `z' = R(bits) · z` — 원본 z를 비트에 따라 회전 (정보 보존)
2. R은 Lie algebra so(n)의 원소 A로부터 Cayley transform으로 생성
3. A는 low-rank 반대칭 행렬 (효율적 + 충분한 표현력)

**파이프라인:**
```
CLIP emb (512D) → Encoder → z ∈ S^99
                               ↓
                   bit_extractor(z) → 100 bits
                               ↓
                   MLP(bits) → U, V → A = UV^T - VU^T (반대칭)
                               ↓
                   R = Cayley(A) ∈ SO(100)
                               ↓
                   z' = R · z  ∈ S^99 (노름 보존, 정보 보존)
                               ↓
                   Decoder → CLIP emb' (512D)
```

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from PIL import Image
import os
from tqdm import tqdm
from collections import defaultdict
import scipy.special
from numbers import Number
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. SO(n) Lie Algebra 배경

### Lie algebra so(n)

SO(n)의 Lie algebra **so(n)**은 **반대칭 행렬**의 공간입니다:
$$\mathfrak{so}(n) = \{A \in \mathbb{R}^{n \times n} : A = -A^T\}$$

반대칭 행렬 A로부터 회전 행렬 R을 생성하는 두 가지 방법:

**Matrix Exponential:** $R = \exp(A) \in SO(n)$
- 정확하지만 계산 비용이 높음

**Cayley Transform:** $R = (I - A)(I + A)^{-1} \in SO(n)$
- A가 반대칭이면 R이 직교 행렬임이 보장
- 선형 시스템 풀기로 계산 → exp보다 효율적

### Low-rank 반대칭 행렬

Full 반대칭 행렬은 $n(n-1)/2 = 4950$개 파라미터가 필요합니다 (n=100).
대신 **low-rank 근사**를 사용합니다:

$$A = UV^T - VU^T, \quad U, V \in \mathbb{R}^{n \times k}$$

- rank(A) ≤ 2k
- 파라미터 수: 2nk (k=10이면 2000개)
- 자동으로 반대칭: $(UV^T - VU^T)^T = VU^T - UV^T = -(UV^T - VU^T)$

### 왜 z에서 출발하는가?

기존: $z' = R(\text{bits}) \cdot \mu$ → z의 연속 정보가 이산 비트로 손실

개선: $z' = R(\text{bits}) \cdot z$ → z의 모든 정보 보존 + 비트로 작은 회전 추가

In [ ]:
class LieAlgebraWatermarker(nn.Module):
    """
    SO(n) Lie Algebra 기반 워터마커
    
    1. z에서 bits 추출
    2. MLP(bits) → low-rank 반대칭 행렬 A
    3. Cayley transform: R = (I-A)(I+A)^{-1} ∈ SO(n)
    4. z' = R · z (노름 보존, 정보 보존)
    """
    
    def __init__(self, dim=100, num_bits=100, rank=10, hidden_dim=256):
        """
        Args:
            dim: latent space 차원
            num_bits: 워터마크 비트 수
            rank: 반대칭 행렬의 rank (2*rank)
            hidden_dim: MLP hidden dimension
        """
        super().__init__()
        self.dim = dim
        self.num_bits = num_bits
        self.rank = rank
        
        # Bit extractor
        self.bit_extractor = nn.Linear(dim, num_bits)
        
        # MLP: bits → U, V for low-rank skew-symmetric A = UV^T - VU^T
        self.uv_predictor = nn.Sequential(
            nn.Linear(num_bits, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2 * dim * rank)
        )
        
        # Initialize to output small values → R ≈ I at start
        with torch.no_grad():
            self.uv_predictor[-1].weight.mul_(0.01)
            self.uv_predictor[-1].bias.zero_()
        
        # Pre-compute identity for Cayley transform
        self.register_buffer('eye', torch.eye(dim))
    
    def extract_bits(self, z, hard=True):
        """z → 100 bits (STE)"""
        logits = self.bit_extractor(z)
        soft_bits = torch.sigmoid(logits)
        
        if hard:
            hard_bits = (soft_bits > 0.5).float()
            bits = hard_bits - soft_bits.detach() + soft_bits
        else:
            bits = soft_bits
        
        return bits, logits
    
    def bits_to_rotation(self, bits):
        """
        bits → R ∈ SO(n) via Lie algebra + Cayley transform
        
        1. MLP(bits) → U, V ∈ R^{n×k}
        2. A = UV^T - VU^T (반대칭, rank ≤ 2k)
        3. R = (I - A)(I + A)^{-1} (Cayley transform)
        """
        batch_size = bits.shape[0]
        bits_symmetric = 2 * bits - 1  # {0,1} → {-1,+1}
        
        # MLP → U, V
        uv = self.uv_predictor(bits_symmetric)  # (batch, 2*dim*rank)
        uv = uv.view(batch_size, 2, self.dim, self.rank)
        U = uv[:, 0]  # (batch, dim, rank)
        V = uv[:, 1]  # (batch, dim, rank)
        
        # A = UV^T - VU^T (반대칭 보장)
        A = torch.bmm(U, V.transpose(-1, -2)) - torch.bmm(V, U.transpose(-1, -2))
        # A: (batch, dim, dim), A = -A^T
        
        # Cayley transform: R = (I - A)(I + A)^{-1}
        I = self.eye.unsqueeze(0).expand(batch_size, -1, -1)
        R = torch.linalg.solve(I + A, I - A)
        # R: (batch, dim, dim), R ∈ SO(n)
        
        return R
    
    def reconstruct(self, bits, z):
        """
        z' = R(bits) · z
        
        z에서 출발하여 비트 정보에 따라 회전
        → z의 모든 연속 정보 보존 + 워터마크 임베딩
        """
        R = self.bits_to_rotation(bits)  # (batch, dim, dim)
        z_recon = torch.bmm(R, z.unsqueeze(-1)).squeeze(-1)  # (batch, dim)
        return z_recon
    
    def forward(self, z, hard=True):
        """
        Full watermarking: z → bits, z' = R(bits) · z
        """
        bits, logits = self.extract_bits(z, hard=hard)
        z_recon = self.reconstruct(bits, z)
        return z_recon, bits, logits


# Quick test
print("Testing LieAlgebraWatermarker...")
wm = LieAlgebraWatermarker(dim=100, num_bits=100, rank=10)
test_z = F.normalize(torch.randn(4, 100), dim=-1)
z_recon, bits, logits = wm(test_z)
print(f"  Input norm:  {test_z.norm(dim=-1)}")
print(f"  Output norm: {z_recon.norm(dim=-1)}")
print(f"  Norm preserved: {torch.allclose(test_z.norm(dim=-1), z_recon.norm(dim=-1), atol=1e-4)}")
print(f"  Cosine sim (z, z'): {F.cosine_similarity(test_z, z_recon, dim=-1)}")
print(f"  Bits shape: {bits.shape}")
print(f"  Watermarker params: {sum(p.numel() for p in wm.parameters()):,}")

# Verify R is orthogonal
R = wm.bits_to_rotation(bits)
RRT = torch.bmm(R, R.transpose(-1, -2))
I = torch.eye(100).unsqueeze(0).expand(4, -1, -1)
print(f"  R is orthogonal: {torch.allclose(RRT, I, atol=1e-4)}")
print(f"  det(R): {torch.linalg.det(R)}")
print("LieAlgebraWatermarker test passed!")

## 3. Complete Pipeline

In [ ]:
class SOLieWatermarking(nn.Module):
    """
    SO(n) Lie Algebra Watermarking
    
    Pipeline:
    1. CLIP 512D → Encoder → z ∈ S^99
    2. bit_extractor(z) → 100 bits
    3. MLP(bits) → A ∈ so(100) → R = Cayley(A) ∈ SO(100)
    4. z' = R · z ∈ S^99
    5. Decoder → CLIP' 512D
    """
    
    def __init__(self, input_dim=512, hidden_dim=256, latent_dim=100, num_bits=100, rank=10):
        super().__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.num_bits = num_bits
        
        # Encoder: CLIP 512D → z ∈ S^{n-1}
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.BatchNorm1d(input_dim),
            nn.ReLU(),
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
        # Decoder: z' → CLIP' 512D
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.BatchNorm1d(input_dim),
            nn.ReLU(),
            nn.Linear(input_dim, input_dim)
        )
        
        # Lie Algebra Watermarker
        self.watermarker = LieAlgebraWatermarker(latent_dim, num_bits, rank, hidden_dim)
    
    def encode(self, x):
        z = self.encoder(x)
        z = F.normalize(z, p=2, dim=-1)
        return z
    
    def decode(self, z):
        x = self.decoder(z)
        x = F.normalize(x, p=2, dim=-1)
        return x
    
    def forward(self, x, labels=None, hard=True):
        z = self.encode(x)
        z_recon, bits, logits = self.watermarker(z, hard=hard)
        x_recon = self.decode(z_recon)
        x_direct = self.decode(z)
        
        return {
            'x_recon': x_recon,
            'x_direct': x_direct,
            'z': z,
            'z_recon': z_recon,
            'bits': bits,
            'logits': logits,
        }


print("Testing SOLieWatermarking...")
model_test = SOLieWatermarking(input_dim=512, latent_dim=100, num_bits=100, rank=10)
test_x = F.normalize(torch.randn(4, 512), dim=-1)
out = model_test(test_x)
print(f"  x_recon shape: {out['x_recon'].shape}")
print(f"  z_recon norm: {out['z_recon'].norm(dim=-1)}")
print(f"  z-z' cosine sim: {F.cosine_similarity(out['z'], out['z_recon'], dim=-1)}")
print(f"  Total params: {sum(p.numel() for p in model_test.parameters()):,}")
print("SOLieWatermarking test passed!")

## 4. Loss Functions

In [ ]:
class SOLieWatermarkingLoss(nn.Module):
    """
    3가지 loss:
    1. Reconstruction: 1 - cos_sim(CLIP, CLIP')
    2. Sphere distance: geodesic_distance(z, z')
    3. Entropy: bit entropy 최소화
    """
    
    def __init__(self, lambda_recon=1.0, lambda_sphere=0.5, lambda_entropy=0.1):
        super().__init__()
        self.lambda_recon = lambda_recon
        self.lambda_sphere = lambda_sphere
        self.lambda_entropy = lambda_entropy
    
    def cosine_similarity_loss(self, x, x_recon):
        cos_sim = F.cosine_similarity(x, x_recon, dim=-1)
        return (1 - cos_sim).mean()
    
    def sphere_distance_loss(self, z, z_recon):
        dot_product = torch.sum(z * z_recon, dim=-1).clamp(-1 + 1e-7, 1 - 1e-7)
        geo_dist = torch.acos(dot_product)
        return geo_dist.mean()
    
    def entropy_loss(self, logits):
        probs = torch.sigmoid(logits)
        entropy = -probs * torch.log(probs + 1e-10) - (1 - probs) * torch.log(1 - probs + 1e-10)
        return entropy.mean()
    
    def forward(self, x, outputs):
        losses = {}
        losses['recon'] = self.cosine_similarity_loss(x, outputs['x_recon'])
        losses['sphere'] = self.sphere_distance_loss(outputs['z'], outputs['z_recon'])
        losses['entropy'] = self.entropy_loss(outputs['logits'])
        
        total = (self.lambda_recon * losses['recon'] +
                 self.lambda_sphere * losses['sphere'] +
                 self.lambda_entropy * losses['entropy'])
        losses['total'] = total
        return losses

## 5. Evaluation Metrics

In [ ]:
class SOLieWatermarkingMetrics:
    
    @torch.no_grad()
    def compute_all_metrics(self, x_original, outputs, labels=None):
        metrics = {}
        
        cos_sim = F.cosine_similarity(x_original, outputs['x_recon'], dim=-1)
        metrics['clip_cosine_sim_mean'] = cos_sim.mean().item()
        metrics['clip_cosine_sim_std'] = cos_sim.std().item()
        metrics['clip_cosine_sim_min'] = cos_sim.min().item()
        metrics['clip_cosine_sim_max'] = cos_sim.max().item()
        
        cos_sim_direct = F.cosine_similarity(x_original, outputs['x_direct'], dim=-1)
        metrics['direct_cosine_sim_mean'] = cos_sim_direct.mean().item()
        
        dot_product = torch.sum(outputs['z'] * outputs['z_recon'], dim=-1).clamp(-1 + 1e-7, 1 - 1e-7)
        geo_dist = torch.acos(dot_product)
        metrics['latent_geodesic_dist_mean'] = geo_dist.mean().item()
        metrics['latent_geodesic_dist_std'] = geo_dist.std().item()
        
        latent_cos_sim = F.cosine_similarity(outputs['z'], outputs['z_recon'], dim=-1)
        metrics['latent_cosine_sim_mean'] = latent_cos_sim.mean().item()
        
        z_recon_norm = outputs['z_recon'].norm(dim=-1)
        metrics['z_recon_norm_mean'] = z_recon_norm.mean().item()
        metrics['z_recon_norm_std'] = z_recon_norm.std().item()
        
        bits = outputs['bits']
        metrics['bit_mean'] = bits.mean().item()
        metrics['bit_std'] = bits.std().item()
        
        probs = torch.sigmoid(outputs['logits'])
        entropy = -probs * torch.log(probs + 1e-10) - (1 - probs) * torch.log(1 - probs + 1e-10)
        metrics['bit_entropy_mean'] = entropy.mean().item()
        
        if labels is not None:
            for class_idx in range(3):
                mask = (labels == class_idx)
                if mask.sum() > 0:
                    metrics[f'class_{class_idx}_cosine_sim'] = cos_sim[mask].mean().item()
        
        return metrics
    
    def print_metrics(self, metrics, title="Evaluation Metrics"):
        print("\n" + "="*60)
        print(title)
        print("="*60)
        
        print("\n[CLIP Embedding Reconstruction]")
        print(f"  Cosine Similarity: {metrics['clip_cosine_sim_mean']:.4f} ± {metrics['clip_cosine_sim_std']:.4f}")
        print(f"  Range: [{metrics['clip_cosine_sim_min']:.4f}, {metrics['clip_cosine_sim_max']:.4f}]")
        print(f"  Direct (no watermark): {metrics['direct_cosine_sim_mean']:.4f}")
        
        print("\n[Latent Space Preservation]")
        print(f"  Geodesic Distance: {metrics['latent_geodesic_dist_mean']:.4f} ± {metrics['latent_geodesic_dist_std']:.4f}")
        print(f"  Cosine Similarity: {metrics['latent_cosine_sim_mean']:.4f}")
        
        print("\n[Orthogonality Verification]")
        print(f"  z_recon L2 Norm: {metrics['z_recon_norm_mean']:.6f} ± {metrics['z_recon_norm_std']:.6f}")
        
        print("\n[Bit Statistics]")
        print(f"  Mean: {metrics['bit_mean']:.4f}")
        print(f"  Std: {metrics['bit_std']:.4f}")
        print(f"  Entropy: {metrics['bit_entropy_mean']:.4f}")
        
        if 'class_0_cosine_sim' in metrics:
            print("\n[Per-Class Cosine Similarity]")
            print(f"  Normal: {metrics.get('class_0_cosine_sim', 'N/A'):.4f}")
            print(f"  Violence: {metrics.get('class_1_cosine_sim', 'N/A'):.4f}")
            print(f"  Sexual: {metrics.get('class_2_cosine_sim', 'N/A'):.4f}")
        
        print("="*60)

## 6. Dataset and CLIP

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Running in Google Colab")
else:
    print("Running locally")

In [ ]:
class SemanticWatermarkDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = os.path.join(root_dir, split)
        self.transform = transform
        self.categories = ['normal', 'violence', 'sexual']
        self.label_to_idx = {cat: idx for idx, cat in enumerate(self.categories)}

        self.image_paths = []
        self.labels = []

        for cat in self.categories:
            cat_dir = os.path.join(self.root_dir, cat)
            if not os.path.exists(cat_dir):
                print(f"Warning: Directory {cat_dir} does not exist")
                continue
            for img_name in os.listdir(cat_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(cat_dir, img_name))
                    self.labels.append(self.label_to_idx[cat])

        print(f"{split} dataset: {len(self.image_paths)} images")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        max_attempts = 10
        for attempt in range(max_attempts):
            try:
                current_idx = (idx + attempt) % len(self.image_paths)
                img_path = self.image_paths[current_idx]
                image = Image.open(img_path).convert('RGB')
                label = self.labels[current_idx]
                if self.transform:
                    image = self.transform(image)
                return image, label
            except Exception as e:
                if attempt == 0:
                    print(f"Warning: Cannot load image {img_path}: {e}")
                continue
        random_idx = np.random.randint(0, len(self.image_paths))
        img_path = self.image_paths[random_idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[random_idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# ============================================================================
# DATASET PATHS CONFIGURATION
# Colab: Place semantic_wm/dataset.zip on Google Drive at
#        /content/drive/MyDrive/semantic_wm/dataset.zip
# Local: Run from the repo root after `preprocessing/prepare_dataset.py`
#        has populated ./dataset/{train,test}/{normal,sexual,violence}/
# ============================================================================
if IN_COLAB:
    zip_path = '/content/drive/MyDrive/semantic_wm/dataset.zip'
    dataset_root = '/content/semantic_wm/dataset'
    if os.path.exists(zip_path) and not os.path.exists('/content/semantic_wm'):
        print("Unzipping dataset...")
        !unzip -q {zip_path} -d /content/semantic_wm/
        print("Done.")
    if os.path.exists(dataset_root):
        print(f"Dataset found at {dataset_root}")
    else:
        print(f"Dataset not found. Please check if {zip_path} exists.")
else:
    dataset_root = './dataset'

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print(f"Dataset root: {dataset_root}")

In [ ]:
print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP model loaded.")

def get_clip_embeddings(images):
    with torch.no_grad():
        inputs = clip_processor(images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        vision_outputs = clip_model.vision_model(**inputs)
        image_embeds = vision_outputs[1]
        image_features = clip_model.visual_projection(image_embeds)
        image_features = F.normalize(image_features, p=2, dim=-1, eps=1e-8)
    return image_features

## 7. Training

In [ ]:
def train_epoch(model, clip_model, train_loader, optimizer, criterion, device):
    model.train()
    epoch_losses = defaultdict(list)
    
    for images, labels in tqdm(train_loader, desc='Training'):
        images = images.to(device)
        labels = labels.to(device)
        
        with torch.no_grad():
            clip_embeddings = get_clip_embeddings(
                [transforms.ToPILImage()(img.cpu()) for img in images]
            )
        
        outputs = model(clip_embeddings, labels=labels, hard=True)
        losses = criterion(clip_embeddings, outputs)
        
        optimizer.zero_grad()
        losses['total'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        for k, v in losses.items():
            epoch_losses[k].append(v.item())
    
    return {k: np.mean(v) for k, v in epoch_losses.items()}


@torch.no_grad()
def evaluate(model, clip_model, test_loader, criterion, metrics_calculator, device):
    model.eval()
    
    all_x = []
    all_outputs = defaultdict(list)
    all_labels = []
    epoch_losses = defaultdict(list)
    
    for images, labels in tqdm(test_loader, desc='Evaluating'):
        images = images.to(device)
        labels = labels.to(device)
        
        clip_embeddings = get_clip_embeddings(
            [transforms.ToPILImage()(img.cpu()) for img in images]
        )
        
        outputs = model(clip_embeddings, labels=labels, hard=True)
        losses = criterion(clip_embeddings, outputs)
        
        for k, v in losses.items():
            epoch_losses[k].append(v.item())
        
        all_x.append(clip_embeddings.cpu())
        for k, v in outputs.items():
            all_outputs[k].append(v.cpu())
        all_labels.append(labels.cpu())
    
    all_x = torch.cat(all_x, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    for k in all_outputs:
        all_outputs[k] = torch.cat(all_outputs[k], dim=0)
    
    metrics = metrics_calculator.compute_all_metrics(all_x, all_outputs, all_labels)
    avg_losses = {k: np.mean(v) for k, v in epoch_losses.items()}
    return avg_losses, metrics, all_outputs, all_labels

In [ ]:
CONFIG = {
    'input_dim': 512,
    'hidden_dim': 256,
    'latent_dim': 100,
    'num_bits': 100,
    'rank': 10,             # low-rank 반대칭 행렬 rank
    'learning_rate': 1e-4,
    'batch_size': 32,
    'num_epochs': 30,
    'lambda_recon': 1.0,
    'lambda_sphere': 0.5,
    'lambda_entropy': 0.1,
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nSkew-symmetric matrix rank: ≤ {2 * CONFIG['rank']}")
print(f"UV params: {2 * CONFIG['latent_dim'] * CONFIG['rank']} (vs full: {CONFIG['latent_dim'] * (CONFIG['latent_dim']-1) // 2})")

In [ ]:
if not os.path.exists(dataset_root):
    raise FileNotFoundError(f"""
    Dataset not found at {dataset_root}
    Colab: Google Drive에 semantic_wm/dataset.zip 업로드 후 재실행
    Local: dataset_root 경로 설정
    """)

print(f"Loading dataset from {dataset_root}")
train_dataset = SemanticWatermarkDataset(dataset_root, split='train', transform=transform)
test_dataset = SemanticWatermarkDataset(dataset_root, split='test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, drop_last=True)

model = SOLieWatermarking(
    input_dim=CONFIG['input_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    latent_dim=CONFIG['latent_dim'],
    num_bits=CONFIG['num_bits'],
    rank=CONFIG['rank']
).to(device)

criterion = SOLieWatermarkingLoss(
    lambda_recon=CONFIG['lambda_recon'],
    lambda_sphere=CONFIG['lambda_sphere'],
    lambda_entropy=CONFIG['lambda_entropy']
)

optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
metrics_calculator = SOLieWatermarkingMetrics()

print(f"\nTotal params: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Watermarker params: {sum(p.numel() for p in model.watermarker.parameters()):,}")
print("Model initialized successfully!")

In [ ]:
train_history = defaultdict(list)
val_history = defaultdict(list)
best_cosine_sim = 0

for epoch in range(CONFIG['num_epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
    
    train_losses = train_epoch(model, clip_model, train_loader, optimizer, criterion, device)
    for k, v in train_losses.items():
        train_history[k].append(v)
    
    val_losses, val_metrics, _, _ = evaluate(
        model, clip_model, test_loader, criterion, metrics_calculator, device
    )
    for k, v in val_losses.items():
        val_history[k].append(v)
    
    print(f"Train Loss: {train_losses['total']:.4f} (recon: {train_losses['recon']:.4f})")
    print(f"Val Loss: {val_losses['total']:.4f} (recon: {val_losses['recon']:.4f})")
    print(f"CLIP Cosine Sim: {val_metrics['clip_cosine_sim_mean']:.4f} ± {val_metrics['clip_cosine_sim_std']:.4f}")
    print(f"Latent Geodesic Dist: {val_metrics['latent_geodesic_dist_mean']:.4f}")
    print(f"z_recon Norm: {val_metrics['z_recon_norm_mean']:.6f}")
    
    if val_metrics['clip_cosine_sim_mean'] > best_cosine_sim:
        best_cosine_sim = val_metrics['clip_cosine_sim_mean']
        torch.save(model.state_dict(), 'best_so_lie_watermark_model.pt')
        print(f"  -> New best model saved! (cosine sim: {best_cosine_sim:.4f})")

print("\nTraining completed!")

## 8. Final Evaluation

In [ ]:
model.load_state_dict(torch.load('best_so_lie_watermark_model.pt'))

print("Final Evaluation on Test Set")
val_losses, val_metrics, all_outputs, all_labels = evaluate(
    model, clip_model, test_loader, criterion, metrics_calculator, device
)

metrics_calculator.print_metrics(val_metrics, "Final Test Metrics (SO(n) Lie Algebra, z-based)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_history['total'], label='Train')
axes[0].plot(val_history['total'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_history['recon'], label='Train')
axes[1].plot(val_history['recon'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Reconstruction Loss (1 - cos_sim)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(train_history['sphere'], label='Train')
axes[2].plot(val_history['sphere'], label='Val')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].set_title('Sphere Distance Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()